<a href="https://colab.research.google.com/github/sinhar1227/deep_learning/blob/main/ANN_Regression_Energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U scikeras scikit-learn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.20.0
    Uninstalling tensorflow-2.20.0:
      Successfully uninstalled tensorflow-2.20.0
ERROR: pip's dependency reso

In [1]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [11]:
df = pd.read_excel('/content/sample_data/energy.xlsx')
df

,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90
...,...,...,...,...,...
9563,16.65,49.69,1014.01,91.00,460.03
9564,13.19,39.18,1023.67,66.78,469.62
9565,31.32,74.33,1012.92,36.48,429.57
9566,24.48,69.45,1013.86,62.39,435.74


In [3]:
X = df.drop('PE', axis=1)
y = df['PE']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.fit_transform(X_test)

In [5]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV

# Function to create model
def create_model(optimizer='adam', neurons=16):

    model = Sequential()

    # Input + Hidden Layer
    model.add(Dense(units=neurons,
                    activation='relu',
                    input_shape=(X_train_scaled.shape[1],)))

    # Output Layer for Regression
    model.add(Dense(units=1))

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model

In [ ]:

# Wrap model for sklearn
model_wrapper = KerasRegressor(
    model=create_model,
    verbose=0
)

# Hyperparameter grid
param_grid = {
    'model__optimizer': ['adam', 'rmsprop'],
    'model__neurons': [8, 16, 32],
    'batch_size': [10, 20],
    'epochs': [10, 50]
}

# Grid Search
grid = GridSearchCV(
    estimator=model_wrapper,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error'
)

# Train
grid_result = grid.fit(X_train_scaled, y_train)

# Results
print("Best Score:", grid_result.best_score_)
print("Best Params:", grid_result.best_params_)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best Score: -19.103202777455678
Best Params: {'batch_size': 10, 'epochs': 50, 'model__neurons': 32, 'model__optimizer': 'adam'}


In [6]:
model = create_model(optimizer='adam', neurons=32)
model.fit(X_train_scaled, y_train, epochs=50, batch_size=10)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 192704.2500 - mae: 438.3847
Epoch 2/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 124920.2734 - mae: 349.2535
Epoch 3/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 54328.2539 - mae: 216.7129
Epoch 4/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 23176.8555 - mae: 130.9646
Epoch 5/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14256.4316 - mae: 102.7772
Epoch 6/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 8792.9609 - mae: 80.4937 
Epoch 7/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4491.0181 - mae: 56.6619
Epoch 8/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1858.5098 - mae: 34.5167
Epoch 9/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 648.4844 - mae: 18.6568
Epoch 10/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 233.9525 - mae: 10.7423
Epoch 11/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 111.0254 - mae: 7.5976
Epoch 12/50
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 69.5

In [7]:
y_train = model.predict(X_train_scaled)
y_test = model.predict(X_test_scaled)

y_traim_mse = mean_squared_error(y_train, y_train)
y_test_mse = mean_squared_error(y_test, y_test)

print(y_traim_mse)
print(y_test_mse)

240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
0.0
0.0
